Создание crop-датасета

In [ ]:
def clean_class_name(name):
    return name.replace(" ", "_").lower()

def crop_image_by_bbox(image, bbox, padding=0.20):
    x, y, w, h = bbox
    img_w, img_h = image.size
    pad_x = w * padding
    pad_y = h * padding
    left = max(int(x - pad_x), 0)
    top = max(int(y - pad_y), 0)
    right = min(int(x + w + pad_x), img_w)
    bottom = min(int(y + h + pad_y), img_h)
    return image.crop((left, top, right, bottom))

def create_crops_from_coco(json_path, images_dir, output_dir, split_name):
    with open(json_path, "r") as f:
        coco = json.load(f)
    id_to_file = {img["id"]: img["file_name"] for img in coco["images"]}
    id_to_class = {cat["id"]: clean_class_name(cat["name"]) for cat in coco["categories"]}
    rows = []
    for ann in tqdm(coco["annotations"], desc=f"Creating {split_name} crops"):
        image_id = ann["image_id"]
        file_name = id_to_file[image_id]
        class_name = id_to_class[ann["category_id"]]
        bbox = ann["bbox"]
        image_path = images_dir / file_name
        if not image_path.exists():
            continue
        image = Image.open(image_path).convert("RGB")
        crop = crop_image_by_bbox(image, bbox)
        class_dir = output_dir / split_name / class_name
        class_dir.mkdir(parents=True, exist_ok=True)
        crop_filename = f"{split_name}_{ann['id']}_{Path(file_name).stem}.jpg"
        crop_path = class_dir / crop_filename
        crop.save(crop_path)
        rows.append({
            "split": split_name,
            "class_name": class_name,
            "crop_filename": crop_filename,
            "crop_path": str(crop_path),
            "bbox_x": bbox[0],
            "bbox_y": bbox[1],
            "bbox_width": bbox[2],
            "bbox_height": bbox[3],
            "bbox_area": bbox[2] * bbox[3]
        })
    return pd.DataFrame(rows)

In [ ]:
if CREATE_CROPS:
    if PROCESSED_DIR.exists():
        shutil.rmtree(PROCESSED_DIR)
    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    split_sources = {
        "train": (ANNOTATION_DIR / "instances_train2017.json", RAW_COCO_DIR / "train2017"),
        "val": (ANNOTATION_DIR / "instances_val2017.json", RAW_COCO_DIR / "val2017"),
        "test": (ANNOTATION_DIR / "instances_test2017.json", RAW_COCO_DIR / "test2017"),
    }
    parts = []
    for split, (json_path, images_dir) in split_sources.items():
        part = create_crops_from_coco(json_path, images_dir, PROCESSED_DIR, split)
        parts.append(part)
    df_crops = pd.concat(parts, ignore_index=True)
    df_crops.to_csv(PROCESSED_DIR / "crops_metadata.csv", index=False)
    if ARCHIVE_PATH.exists():
        ARCHIVE_PATH.unlink()
    print("Crop-датасет создан:", len(df_crops))
else:
    print("Используем уже готовый crop-датасет.")

Используем уже готовый crop-датасет.


Загрузка кропов из диска

In [ ]:
metadata_path = PROCESSED_DIR / "crops_metadata.csv"
needed_folders = [PROCESSED_DIR / "train", PROCESSED_DIR / "val", PROCESSED_DIR / "test"]
if not metadata_path.exists() or not all(folder.exists() for folder in needed_folders):
    raise FileNotFoundError("Готовый crop-датасет не найден")
print("Готовый crop-датасет найден")
if not ARCHIVE_PATH.exists():
    print("Создаём архив crop-датасета")
    with tarfile.open(ARCHIVE_PATH, "w:gz") as tar:
        tar.add(PROCESSED_DIR, arcname="cardd_crops")
else:
    print("Архив crop-датасета уже есть")
if LOCAL_DATA_DIR.exists():
    shutil.rmtree(LOCAL_DATA_DIR)
with tarfile.open(ARCHIVE_PATH, "r:gz") as tar:
    tar.extractall(path="/content")
DATA_DIR = LOCAL_DATA_DIR

Готовый crop-датасет найден
Архив crop-датасета уже есть


/tmp/ipykernel_1682/898443679.py:15: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path="/content")


In [ ]:
df_crops = pd.read_csv(metadata_path)
print("Столбцы в metadata:", df_crops.columns.tolist())
if "crop_filename" not in df_crops.columns:
    df_crops["crop_filename"] = df_crops["crop_path"].apply(lambda x: Path(x).name)
df_crops["local_path"] = df_crops.apply(lambda row: str(DATA_DIR / row["split"] / row["class_name"] / row["crop_filename"]),axis=1)
print("Всего crop-изображений:", len(df_crops))
df_crops.head()

Столбцы в metadata: ['split', 'source_image', 'annotation_id', 'class_name', 'bbox_x', 'bbox_y', 'bbox_w', 'bbox_h', 'crop_path']
Всего crop-изображений: 8740


,split,source_image,annotation_id,class_name,bbox_x,bbox_y,bbox_w,bbox_h,crop_path,crop_filename,local_path
0,train,000001.jpg,1,scratch,167.04,40.21,202.79,131.34,/content/drive/MyDrive/car_damage_project/proc...,000001_ann_1_scratch.jpg,/content/cardd_crops/train/scratch/000001_ann_...
1,train,000001.jpg,2,tire_flat,160.66,112.45,684.19,551.02,/content/drive/MyDrive/car_damage_project/proc...,000001_ann_2_tire_flat.jpg,/content/cardd_crops/train/tire_flat/000001_an...
2,train,000002.jpg,3,tire_flat,476.73,241.02,397.20,346.68,/content/drive/MyDrive/car_damage_project/proc...,000002_ann_3_tire_flat.jpg,/content/cardd_crops/train/tire_flat/000002_an...
3,train,000003.jpg,4,tire_flat,354.29,0.00,645.71,521.12,/content/drive/MyDrive/car_damage_project/proc...,000003_ann_4_tire_flat.jpg,/content/cardd_crops/train/tire_flat/000003_an...
4,train,000004.jpg,5,tire_flat,129.29,0.00,718.62,649.69,/content/drive/MyDrive/car_damage_project/proc...,000004_ann_5_tire_flat.jpg,/content/cardd_crops/train/tire_flat/000004_an...
